# Data Agent 评测教程：以电商问数为例

这本 notebook 把 `evals/` 目录的评测体系从头讲一遍，并用 **glm-5.3-flash 的真实评测结果**做解读。读完你应该能回答：

1. 一个 NL2SQL Agent 的“答对”该怎么定义，为什么比较**执行结果**而不是 SQL 文本
2. 评测题怎么出才有区分度：每道题都有一条“典型错误写法”，并且要证明它真的能被判错
3. 答错时怎么知道错在哪一环：召回、过滤，还是 SQL 写错
4. 怎么用评测做回归门禁和消融实验：改了 Prompt、关掉某个模块，准确率变了多少
5. 多轮对话怎么评：teacher forcing 和自由对话分别衡量什么

**怎么读：** 第 1～5 节只需要 MySQL 里有数仓数据；第 6～7 节会真的让 Agent 答一道题，需要全部服务（MySQL、ES、Qdrant、TEI）、构建好的知识库和 LLM key；第 9～10 节读取已提交的评测结果，不需要任何服务。

| 前提 | 命令 |
| --- | --- |
| 安装依赖 | `uv sync --group notebook` |
| 启动服务 | `docker compose -f docker/docker-compose.yaml up -d` |
| 写入数仓数据 | `uv run python -m app.scripts.generate_dw_data`（或导入 `docker/snapshot/dw_snapshot.sql.gz`） |
| 构建知识库 | `uv run python -m app.scripts.build_meta_knowledge -c conf/meta_config.yaml` |
| LLM key | 项目根目录 `.env` 里写 `LLM_API_KEY=...`（智谱开放平台） |

Windows + Docker 与 Linux 的差异见文末附录。

## 0. 准备

In [1]:
import inspect
import json
import os
import sys
from pathlib import Path

import pandas as pd
import yaml

# 项目根目录：notebook 在 notebooks/ 下运行时取上一级
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # 配置里的日志目录等相对路径都以项目根目录为准
sys.path.insert(0, str(ROOT))

# 导入项目代码前先把控制台日志调到 WARNING，否则每个节点的 INFO 日志会刷屏
from app.conf.app_config import app_config  # noqa: E402

app_config.logging.console.level = "WARNING"

from evals.compare import results_equal  # noqa: E402
from evals.dataset import (  # noqa: E402
    DEFAULT_DATASET,
    MULTITURN_DATASET,
    check_leakage,
    check_structure,
    load_dataset,
    load_examples,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

traps = load_dataset(DEFAULT_DATASET)
multiturn = load_dataset(MULTITURN_DATASET)
print(f"单轮题 {len(traps)} 道，多轮对话 {len(multiturn)} 段")

单轮题 50 道，多轮对话 22 段


## 1. 评测要回答什么

一次评测要回答四个问题：

| 问题 | 指标 | 代码 |
| --- | --- | --- |
| 答对了吗 | 执行准确率（EX）：模型 SQL 与标准 SQL 的执行结果等价的比例 | `evals/compare.py` |
| 错在哪 | 失败环节：流程异常 → 召回遗漏 → 过滤误删 → SQL 不可执行 → 结果不符（多轮再加“意图错误”） | `evals/scoring.py` |
| 代价多大 | 每题 LLM 调用次数、token 数、耗时、SQL 修正次数 | `evals/runner.py` |
| 变差了吗 | 与基线比较：必过题失败、准确率下降超过阈值、成本上升超过阈值 | `scoring.compare_with_baseline` |

整体流程：

```
评测集 YAML ──► 答题 (predictor: agent / gold / trap)
                  │  agent：跑问数图，捕获每个节点的输出
                  ▼
        执行预测 SQL 与标准 SQL ──► 判分 results_equal ──► 归因 attribute_failure
                                                              │
                         results.jsonl / summary.json / summary.md ◄── 汇总 summarize
                                                              │
                                                  与基线比较（退化时退出码 1）
```

文件地图：

| 文件 | 职责 | 需要数据库 |
| --- | --- | --- |
| `evals/datasets/traps.yaml` | 单轮题：10 类业务口径陷阱 × 5 题 | |
| `evals/datasets/multiturn.yaml` | 多轮题：6 类追问 × 20 段对话 | |
| `evals/dataset.py` | 静态检查：字段齐全、expect 与 SQL 一致、只读、与 few-shot 示例不泄漏（pytest 会跑） | 否 |
| `evals/check_dataset.py` | 在库检查：标准 SQL 有结果、错误写法的结果与标准答案不同 | 是 |
| `evals/compare.py` | 结果等价判定（纯函数） | 否 |
| `evals/scoring.py` | 归因、汇总、基线比较、报告（纯函数） | 否 |
| `evals/runner.py` | 单轮答题与打分的命令行入口 | 是 |
| `evals/multiturn.py` | 多轮答题（teacher / free） | 是 |
| `evals/llm_cache.py` | LLM 调用录制回放（SQLite） | 否 |

## 2. 评测集：题目长什么样

这套题不考“会不会写 SQL”，考的是**业务口径**：每道题都对应一种“SQL 能跑、结果却错了”的陷阱，比如把未支付订单算进销售额、订单头金额关联明细后重复累加、把收货地当成用户常住地。

每道题的字段：

| 字段 | 作用 |
| --- | --- |
| `question` / `paraphrases` | 问题和其他问法（测“换个说法还能不能答对”） |
| `gold_sql` | 标准 SQL，判分时比较它的执行结果 |
| `trap_sql` / `trap` | 这类陷阱的典型错误写法和错在哪。它让题目“可证伪”：如果错误写法也能得到同样结果，这道题就测不出这个陷阱 |
| `expect` | 回答本题需要的表 / 指标 / 取值，用于答错时判断是不是召回或过滤阶段漏了 |
| `order_matters` | 只有问题本身要求排序时才比较行顺序 |
| `today` | 注入给 Agent 的“今天”，让“本月”“去年同期”可复现 |
| `must_pass` | 核心题，失败即视为回归 |

In [2]:
summary = pd.DataFrame(
    [
        {
            "类别": item["category"],
            "题目": item["id"],
            "难度": item["difficulty"],
            "必过": bool(item.get("must_pass")),
            "问题": item["turns"][0]["question"],
        }
        for item in traps
    ]
)
summary.groupby("类别").agg(
    题数=("题目", "count"), 必过=("必过", "sum"), 示例=("问题", "first")
)

,题数,必过,示例
类别,,,
activity,5,1,2025 年双 11 活动的商品一共卖了多少件？
date_role,5,0,2025 年 11 月 11 日当天完成支付的订单有多少笔？
distinct,5,1,2025 年有多少个支付买家？
fanout,5,1,2025 年各一级类目的销售额是多少？
funnel,5,1,2025 年 12 月的浏览转化率是多少？
order_vs_pay,5,1,2025 年的销售额是多少？
refund,5,1,2025 年支付的订单，扣除退款成功的金额后，净销售额是多少？
region_role,5,1,2025 年发往上海市的支付订单有多少笔？
relative_time,5,1,本月的销售额是多少？


看一道完整的题：`fanout_01` 考的是“订单头金额关联明细后求和会被重复计算”。

In [3]:
item = next(i for i in traps if i["id"] == "fanout_01")
print(yaml.dump(item, allow_unicode=True, sort_keys=False))

id: fanout_01
category: fanout
tags:
- 多表
- 雪花
- 指标
- 分组
- 支付口径
difficulty: medium
today: '2025-12-31'
must_pass: true
turns:
- question: 2025 年各一级类目的销售额是多少？
  paraphrases:
  - 今年每个大类卖了多少钱
  gold_sql: "select c.category_l1 as 一级类目, sum(i.pay_amount) as 销售额\nfrom fact_order_item\
    \ i\njoin fact_order o on i.order_id = o.order_id\njoin dim_sku k on i.sku_id\
    \ = k.sku_id\njoin dim_category c on k.category_id = c.category_id\nwhere o.pay_time\
    \ is not null\n  and o.pay_date_id between 20250101 and 20251231\ngroup by c.category_l1\n"
  trap_sql: "select c.category_l1 as 一级类目, sum(o.order_amount) as 销售额\nfrom fact_order\
    \ o\njoin fact_order_item i on i.order_id = o.order_id\njoin dim_sku k on i.sku_id\
    \ = k.sku_id\njoin dim_category c on k.category_id = c.category_id\nwhere o.pay_time\
    \ is not null\n  and o.pay_date_id between 20250101 and 20251231\ngroup by c.category_l1\n"
  trap: 订单金额关联明细后求和，一笔多件商品的订单被重复计算
  expect:
    tables:
    - fact_order
    - fact_orde

## 3. 为什么比较执行结果，而不是 SQL 文本

同一个问题可以有很多种正确写法：按支付日期筛时间，可以直接写 `pay_date_id BETWEEN ...`，也可以关联 `dim_date` 按 `year` 筛；列名可以叫“销售额”也可以叫 `gmv`。逐字比较 SQL 会把正确答案判错。

所以主流的 NL2SQL 基准（Spider、BIRD）都用**执行准确率（Execution Accuracy, EX）**：两条 SQL 在同一个库上执行，结果等价就算对。代价是需要一个固定的数据库，并且要定义清楚“结果等价”（下一节）。

下面在数仓上验证：两种正确写法结果相同，错误写法结果不同。

In [4]:
from app.clients.mysql_client_manager import dw_mysql_client_manager
from evals.runner import execute  # runner 用的同一个执行函数：只读校验后执行

dw_mysql_client_manager.init()

gold = item["turns"][0]["gold_sql"]
# 另一种正确写法：关联日期维度按年筛选，列名也不同
variant = """
select c.category_l1 as cat, sum(i.pay_amount) as gmv
from fact_order_item i
join fact_order o on i.order_id = o.order_id
join dim_date d on o.pay_date_id = d.date_id
join dim_sku k on i.sku_id = k.sku_id
join dim_category c on k.category_id = c.category_id
where o.pay_time is not null and d.year = 2025
group by c.category_l1
"""
trap = item["turns"][0]["trap_sql"]

gold_rows = await execute(gold)
variant_rows = await execute(variant)
trap_rows = await execute(trap)
print("另一种正确写法判为相同：", results_equal(variant_rows, gold_rows))
print("错误写法判为相同：", results_equal(trap_rows, gold_rows))
pd.DataFrame(gold_rows).merge(
    pd.DataFrame(trap_rows), on="一级类目", suffixes=("_标准", "_错误写法")
)

/home/user/shopkeeper-agent/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


另一种正确写法判为相同： True
错误写法判为相同： False


,一级类目,销售额_标准,销售额_错误写法
0,美妆个护,4699473.41,8574239.25
1,服饰鞋包,9239559.60,16595022.03
2,食品饮料,964527.47,1720710.92
3,手机数码,17505016.31,26477118.47
4,家用电器,10865482.65,16329393.84
5,母婴,1505125.67,2665676.78
6,家居日用,1645673.67,2921886.48


## 4. 判分规则逐条拆解（`evals/compare.py`）

`results_equal(pred, gold, order_matters)` 判断两个“列名 → 值”的字典列表是否等价，规则参照 Spider 2.0 的宽松口径：

1. **忽略列名和列顺序**：标准答案的每一列，都要在预测结果里找到一列内容相同的列（列名是模型起的，不该影响对错）
2. **允许多出列**：问“哪个大区最高”，模型顺手带上销售额也算对；但不能少列、行数必须一致
3. **行顺序**：只有 `order_matters` 时才逐行比较，否则当多重集合比较
4. **数值容差**：相对误差 1e-9（只吸收浮点误差，金额动辄几百万，放宽会放过几元的差异）；预测值四舍五入到 1～4 位小数也算对；同一列整体是标准值的 100 倍也算对（比率 vs 百分数）
5. **类型归一**：Decimal、整数统一转 float，日期转 ISO 字符串，字符串去掉首尾空白
6. **时间标签换算**：一列预测值整体是同一年的“年月”标签（`'2025-01'`、`202501`、`'2025年1月'`）时，可以按月份数字与标准列比较；日期 `'2025-12-25'` 与日期键 `20251225` 互相认可。第 6 条是看了真实评测结果之后加的，过程见第 9 节

实现上分两步：先为每个标准列找出所有“内容能对上”的预测列（含倍数和时间换算），再回溯搜索一组互不重复、并且逐行也能对齐的对应关系。第二步是必要的：两列各自都能对上，不代表它们在同一行上是配对的。

In [5]:
G = [{"area": "华东", "amt": 1234.5678}, {"area": "华南", "amt": 999.1}]
cases = [
    (
        "列名不同",
        [{"大区": "华东", "销售额": 1234.5678}, {"大区": "华南", "销售额": 999.1}],
        False,
    ),
    (
        "列顺序不同",
        [{"amt": 1234.5678, "area": "华东"}, {"amt": 999.1, "area": "华南"}],
        False,
    ),
    (
        "多出一列",
        [
            {"area": "华东", "amt": 1234.5678, "n": 3},
            {"area": "华南", "amt": 999.1, "n": 1},
        ],
        False,
    ),
    ("少一列", [{"area": "华东"}, {"area": "华南"}], False),
    (
        "四舍五入到 2 位",
        [{"area": "华东", "amt": 1234.57}, {"area": "华南", "amt": 999.1}],
        False,
    ),
    (
        "四舍五入但值错了",
        [{"area": "华东", "amt": 1234.58}, {"area": "华南", "amt": 999.1}],
        False,
    ),
    (
        "差 1 分钱",
        [{"area": "华东", "amt": 1234.5778}, {"area": "华南", "amt": 999.1}],
        False,
    ),
    (
        "行顺序不同（不要求顺序）",
        [{"area": "华南", "amt": 999.1}, {"area": "华东", "amt": 1234.5678}],
        False,
    ),
    (
        "行顺序不同（要求顺序）",
        [{"area": "华南", "amt": 999.1}, {"area": "华东", "amt": 1234.5678}],
        True,
    ),
    (
        "字符串带空格",
        [{"area": " 华东 ", "amt": 1234.5678}, {"area": "华南", "amt": 999.1}],
        False,
    ),
    (
        "行列错配（列都能对上，行对不上）",
        [{"area": "华东", "amt": 999.1}, {"area": "华南", "amt": 1234.5678}],
        False,
    ),
]
rows = [
    {
        "情形": name,
        "order_matters": om,
        "判定": "相等" if results_equal(pred, G, om) else "不等",
    }
    for name, pred, om in cases
]
# 时间标签：标准答案按 dim_date.month 输出 1、2，模型常输出带年份的月份
M = [{"月份": 1, "销售额": 10.5}, {"月份": 2, "销售额": 20.0}]
for name, labels in [
    ("月份写成 '2025-01'", ["2025-01", "2025-02"]),
    ("月份写成 202501", [202501, 202502]),
    ("跨年的月份标签（不换算）", ["2024-01", "2025-02"]),
]:
    pred = [{"m": labels[0], "amt": 10.5}, {"m": labels[1], "amt": 20.0}]
    rows.append(
        {
            "情形": name,
            "order_matters": False,
            "判定": "相等" if results_equal(pred, M) else "不等",
        }
    )
# 比率 vs 百分数
rows.append(
    {
        "情形": "比率写成百分数（×100）",
        "order_matters": False,
        "判定": "相等" if results_equal([{"r": 12.5}], [{"rate": 0.125}]) else "不等",
    }
)
pd.DataFrame(rows)

,情形,order_matters,判定
0,列名不同,False,相等
1,列顺序不同,False,相等
2,多出一列,False,相等
3,少一列,False,不等
4,四舍五入到 2 位,False,相等
5,四舍五入但值错了,False,不等
6,差 1 分钱,False,不等
7,行顺序不同（不要求顺序）,False,相等
8,行顺序不同（要求顺序）,True,不等
9,字符串带空格,False,相等


**宽松是有代价的**：规则越宽，越可能把错误答案判对。例如错误写法的结果恰好是标准答案的 100 倍、或者四舍五入后恰好相同。所以第 5 节的自检要求：每道题的错误写法，在“保留两位小数”之后仍然要能和标准答案区分开。

## 5. 评测集自检：测不出错的题没有意义

一道题如果错误写法也能“答对”，它就测不出它声称的陷阱。自检分两层：

- **静态检查**（`evals/dataset.py`，pytest 每次都跑）：必填字段齐全、id 唯一、每类题数、`expect.tables` 都出现在标准 SQL 里、`expect.metrics` 都是已定义的指标、两条 SQL 都能通过只读校验，以及**防泄漏**：few-shot 示例和评测题的问题相似度 ≥ 0.8、或 SQL 相似度 ≥ 0.9 就报错（示例会被召回进 Prompt，太像评测题分数就会虚高）
- **在库检查**（`evals/check_dataset.py`）：标准 SQL 结果非空；错误写法能执行，结果与标准答案不同，并且四舍五入到两位小数后仍然不同

改了数据生成器、数据规模或题目后都要重跑在库检查：`uv run python -m evals.check_dataset`

In [6]:
print("单轮静态检查问题：", check_structure(traps, per_category=5))
print("多轮静态检查问题：", check_structure(multiturn, per_category=None))
print("示例泄漏：", check_leakage(traps + multiturn, load_examples()))

单轮静态检查问题： []


多轮静态检查问题： []


示例泄漏： []


在库检查的核心逻辑，对每类挑一题演示（完整版见 `check_dataset.check_results`）：

In [7]:
from evals.check_dataset import _rounded

rows = []
for category in summary["类别"].unique():
    it = next(i for i in traps if i["category"] == category)
    turn = it["turns"][0]
    g = await execute(turn["gold_sql"])
    t = await execute(turn["trap_sql"])
    om = bool(turn.get("order_matters"))
    rows.append(
        {
            "题目": it["id"],
            "标准结果行数": len(g),
            "错误写法与标准相同": results_equal(t, g, om),
            "两位小数后相同": results_equal(_rounded(t), g, om),
            "陷阱": turn["trap"],
        }
    )
pd.DataFrame(rows)

,题目,标准结果行数,错误写法与标准相同,两位小数后相同,陷阱
0,order_vs_pay_01,1,False,False,没有排除未支付和已取消的订单，算成了下单金额
1,fanout_01,7,False,False,订单金额关联明细后求和，一笔多件商品的订单被重复计算
2,refund_01,1,False,False,扣除了全部退款申请，包括被拒绝、已撤销和处理中的
3,date_role_01,1,False,False,用下单日期代替支付日期，漏掉 10 号下单 11 号支付的，多算 11 号下单 12 号支付的
4,region_role_01,1,False,False,按用户常住城市统计，而问题问的是收货城市
5,snowflake_01,1,False,False,成本价是单件成本，没有乘以件数
6,distinct_01,1,False,False,没有去重，统计成了支付订单数
7,activity_01,1,False,False,用活动期间代替活动归因，把期间内未参加活动的商品也算了进去
8,relative_time_01,1,False,False,把“本月”理解成最近 30 天
9,funnel_01,1,False,False,分母用了浏览次数而不是浏览用户数


## 6. 让 Agent 答一道题，看它每一步做了什么

`--predictor agent` 时，runner 调用 `run_agent(question, today)`：

- 用 `graph.astream(stream_mode=["updates", "custom"])` 跑问数图。`updates` 模式会在每个节点执行完后给出它返回的 dict，runner 把它们按节点名存下来，**归因就靠这些中间结果**
- 通过 LangChain 回调 `UsageCallback` 统计 LLM 调用次数和 token
- 最终 SQL 取 `correct_sql` 的输出（修正过），没有修正过就取 `generate_sql` 的输出

另外两种 predictor 用来**检验评测系统本身**：`gold` 直接用标准 SQL 作答，必须是 100%；`trap` 用错误写法作答，必须是 0%。任何一个不符合，说明判分或题目有问题，先修评测再看模型。

下面这个单元会真正调用 LLM（约 6 次）。设置了和评测相同的 LLM 缓存：如果评测时录制过这道题，这里会直接回放，不花钱、结果也和评测一致。

In [8]:
from langchain_core.globals import set_llm_cache

from app.clients.embedding_client_manager import embedding_client_manager
from app.clients.es_client_manager import es_client_manager
from app.clients.mysql_client_manager import meta_mysql_client_manager
from app.clients.qdrant_client_manager import qdrant_client_manager
from evals.llm_cache import SQLiteLLMCache
from evals.runner import available_items, run_agent
from evals.scoring import missing_items

for client in (
    meta_mysql_client_manager,
    qdrant_client_manager,
    es_client_manager,
    embedding_client_manager,
):
    client.init()
cache = SQLiteLLMCache(ROOT / "evals" / ".cache" / "glm-5.3-flash.sqlite")
set_llm_cache(cache)

item = next(i for i in traps if i["id"] == "region_role_01")
turn = item["turns"][0]
answer = await run_agent(turn["question"], item["today"])
print("问题：", turn["question"])
print(
    f"LLM 调用 {answer['llm_calls']} 次（缓存命中 {cache.hits}，未命中 {cache.misses}），耗时 {answer['seconds']:.1f} 秒"
)
print("经过的节点：", list(answer["outputs"]))
print("\n模型 SQL：\n" + answer["sql"])
print("\n标准 SQL：\n" + turn["gold_sql"])

Building prefix dict from the default dictionary ...


Loading model from cache /tmp/jieba.cache


Loading model cost 0.589 seconds.


Prefix dict has been built successfully.


/home/user/shopkeeper-agent/evals/llm_cache.py:48: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return loads(row[0])


/* select#1 */ select count(`dw`.`f`.`order_id`) AS `支付订单数` from `dw`.`fact_order` `f` join `dw`.`dim_region` `r` where ((`dw`.`f`.`receiver_region_id` = `dw`.`r`.`region_id`) and (`dw`.`f`.`pay_time` is not null) and (`dw`.`f`.`order_time` >= TIMESTAMP'2025-01-01 00:00:00') and (`dw`.`f`.`order_time` < TIMESTAMP'2026-01-01 00:00:00') and (`dw`.`r`.`city` = '上海市'))


问题： 2025 年发往上海市的支付订单有多少笔？
LLM 调用 6 次（缓存命中 6，未命中 0），耗时 2.8 秒
经过的节点： ['extract_keywords', 'recall_value', 'recall_knowledge', 'recall_metric', 'recall_column', 'merge_retrieved_info', 'filter_metric', 'filter_table', 'add_extra_context', 'generate_sql', 'validate_sql', 'run_sql']

模型 SQL：
SELECT COUNT(f.order_id) AS 支付订单数
FROM fact_order f
JOIN dim_region r ON f.receiver_region_id = r.region_id
WHERE f.pay_time IS NOT NULL
  AND f.order_time >= '2025-01-01' AND f.order_time < '2026-01-01'
  AND r.city = '上海市'

标准 SQL：
select count(*) as 支付订单数
from fact_order o
join dim_region r on o.receiver_region_id = r.region_id
where r.city = '上海市'
  and o.pay_time is not null
  and o.pay_date_id between 20250101 and 20251231



In [9]:
recalled, filtered = available_items(answer["outputs"])
expect = turn["expect"]
pd.DataFrame(
    {
        "期望": {k: sorted(expect.get(k, [])) for k in ("tables", "metrics", "values")},
        "召回到": {
            k: sorted(recalled.get(k, set())) for k in ("tables", "metrics", "values")
        },
        "过滤后": {
            k: sorted(filtered.get(k, set())) for k in ("tables", "metrics", "values")
        },
    }
)

,期望,召回到,过滤后
tables,"[dim_region, fact_order]","[dim_activity, dim_date, dim_region, dim_shop, dim_sku, dim_user, fact_order, fact_order_item, fact_refund, fact_use...","[dim_region, fact_order]"
metrics,[],"[下单订单数, 下单金额, 件单价, 净支付金额, 复购率, 客单价, 支付买家数, 支付订单数, 支付金额, 笔单价]",[支付订单数]
values,[上海市],"[2024年38女王节, 2024年618大促, 2024年双11, 2024年双12, 2024年货节, 2025年38女王节, 2025年618大促, 2025年双11, 2025年双12, 2025年货节, 上海市, 已支付,...",[]


In [10]:
pred_rows = await execute(answer["sql"])
gold_rows = await execute(turn["gold_sql"])
print("召回遗漏：", missing_items(expect, recalled))
print(
    "过滤误删：",
    missing_items({k: v for k, v in expect.items() if k != "values"}, filtered),
)
print(
    "结果等价：", results_equal(pred_rows, gold_rows, bool(turn.get("order_matters")))
)

召回遗漏： []
过滤误删： []
结果等价： True


**结果相同不等于口径正确。** 仔细看上面的模型 SQL：如果它是按下单时间（`order_time`）筛选年份的，严格说口径不对，“支付订单”应该按支付日期筛选；但上海市恰好没有“跨年支付”的订单，两种写法在这份数据上结果相同，于是判为正确。这是执行准确率的固有局限：**只能发现在这份数据上会产生差异的错误。** 评测集里的 `date_role` 类题目就是专门构造了跨日、跨月支付的订单，来考这一点的。

（这里回放出的 SQL 可能与 `evals/results/` 里记录的不同：TEI 的批次噪声让同一道题在不同运行里拼出了两个版本的提示词，缓存里两个版本都有。见第 9 节“关于可复现”。）

## 7. 失败归因：错在哪一环

答错时，按流水线顺序找**第一个**出问题的环节（借鉴 Uber QueryGPT 按意图、选表、生成分别统计的做法）：

In [11]:
from evals import scoring

print(inspect.getsource(scoring.attribute_failure))

def attribute_failure(
    *,
    error: str | None,
    sql: str | None,
    recall_missing: list[str] | None,
    filter_missing: list[str] | None,
    sql_executable: bool,
) -> str:
    """答错时按流水线顺序找出第一个出问题的环节"""
    if error and not sql:
        return STAGE_ERROR
    if recall_missing:
        return STAGE_RECALL
    if filter_missing:
        return STAGE_FILTER
    if not sql or not sql_executable:
        return STAGE_INVALID_SQL
    return STAGE_WRONG_RESULT



为什么按这个顺序：

- **召回遗漏**排在最前：需要的表或指标在召回阶段就没拿到，后面的过滤和生成不可能补回来，这时优化 Prompt 是白费力气
- **过滤误删**：召回到了，但 `filter_table` / `filter_metric` 把它删了
- **SQL 不可执行**：上下文齐全，但 SQL 修正次数用完仍然报错
- **结果不符**：能执行但结果错，通常是口径或逻辑错误（例如用了下单口径、关联出扇出）

几个需要注意的局限：

- 召回和过滤的判断完全依赖题目里人工标注的 `expect`，标注不全就会把问题归到后面的环节
- 取值（`values`）只检查召回、不检查过滤，因为过滤节点只输出表和字段
- “结果不符”是兜底类别，里面混着各种逻辑错误，需要看失败明细里的 SQL 才能细分
- 多轮评测在最前面再加一个“意图错误”：意图判错且答错的轮次优先归到这里

归因的价值在于指导优化方向：如果大部分失败是召回遗漏，就该优化检索词扩展、阈值或别名，而不是去调 SQL 生成的 Prompt。

## 8. 跑一次完整评测

完整评测耗时较长（单轮 50 题约 15 分钟），建议在终端里跑，notebook 只用来读结果。以下命令以 Linux / macOS 的 bash 为准，Windows 的写法见附录。

```bash
uv run python -m evals.runner --predictor gold        # 自检：必须 100%
uv run python -m evals.runner --predictor trap        # 自检：必须 0%
uv run python -m evals.runner --paraphrases --llm-cache evals/.cache/glm-5.3-flash.sqlite \
    --baseline evals/baselines/glm-5.3-flash.json --save-baseline       # 基线（含换问法）
uv run python -m evals.runner --no-examples --llm-cache evals/.cache/glm-5.3-flash.sqlite        # 消融：去掉 few-shot 示例
uv run python -m evals.runner --no-semantic-check --llm-cache evals/.cache/glm-5.3-flash.sqlite  # 消融：去掉语义检查
uv run python -m evals.runner --baseline evals/baselines/glm-5.3-flash.json                       # 与基线比较，退化时退出码 1
uv run python -m evals.runner --multiturn teacher --llm-cache evals/.cache/glm-5.3-flash.sqlite  # 多轮
```

**LLM 缓存**（`evals/llm_cache.py`）：以“模型参数 + 完整提示词”为键，把模型回答存进 SQLite。重跑时提示词没变就直接回放，不花钱、结果完全可复现；只要提示词有一个字变了（改了 Prompt、知识库、召回结果），对应的调用就会真实请求一次。做消融实验时，没受影响的调用都能命中缓存，成本很低。

**可复现的前提：提示词必须逐字节确定。** 缓存的键是完整提示词，只要每次运行拼出来的提示词有一点不同，就只能真实请求，结果也可能不同。这个项目踩过一次坑：`extract_keywords` 和三个召回节点原来用 `set` 给检索词去重，而 Python 每次启动时 set 的顺序都可能不同（字符串哈希带随机种子），检索顺序一变，召回结果的顺序、合并后的表结构顺序、过滤和生成 SQL 的提示词就都跟着变。表现是：换一个进程重跑同一道题，6 次 LLM 调用只有检索词扩展那 3 次命中缓存。改成 `dict.fromkeys` 保序去重后，用不同的 `PYTHONHASHSEED` 在两个进程里各跑一次，第二次 12 次调用全部命中。凡是会进入提示词的列表，都不要用 `set` 决定顺序。

**基线与回归门禁**（`scoring.compare_with_baseline`）：判为退化的条件是任一必过题失败、整体准确率下降超过 2 个百分点（`--max-accuracy-drop`），或平均 LLM 调用次数上升超过 30%。退化时退出码为 1，可以直接接进 CI。

**报告**写到 `evals/reports/<时间戳>/`（已 gitignore）：`results.jsonl` 每题一行明细（SQL、错误、归因、成本），`summary.json` 汇总，`summary.md` 可读报告。本仓库把 glm-5.3-flash 的一次完整评测结果复制到了 `evals/results/glm-5.3-flash/`，下面两节读的就是它。

## 9. 解读真实结果（glm-5.3-flash，单轮）

`RESULTS` 指向已提交的结果；想读自己跑出来的结果，把它改成 `evals/reports/<时间戳>` 所在的目录并按同样的子目录名放好即可。

In [12]:
RESULTS = ROOT / "evals" / "results" / "glm-5.3-flash"


def load_run(name):
    """读取一次评测的汇总和逐题明细"""
    folder = RESULTS / name
    s = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
    r = pd.DataFrame(
        [
            json.loads(line)
            for line in (folder / "results.jsonl")
            .read_text(encoding="utf-8")
            .splitlines()
        ]
    )
    return s, r


runs = {name: load_run(name) for name in ("baseline", "no_examples", "no_lint")}
pd.DataFrame(
    {
        name: {
            "题数": s["total"],
            "准确率": f"{s['accuracy']:.1%}",
            "平均 LLM 调用": s["avg_llm_calls"],
            "平均 tokens": s["avg_tokens"],
            "平均耗时(秒)": s["avg_seconds"],
            "必过题失败": len(s["must_pass_failed"]),
        }
        for name, (s, _) in runs.items()
    }
)

,baseline,no_examples,no_lint
题数,100,50,50
准确率,91.0%,96.0%,96.0%
平均 LLM 调用,6.05,6.12,6.0
平均 tokens,19577.9,19771.0,18560.3
平均耗时(秒),17.12,8.36,12.89
必过题失败,1,1,1


- `baseline`：完整配置，每题原问法 + 一种换问法，共 100 次作答
- `no_examples`：关掉已验证问答示例（few-shot），只答原问法
- `no_lint`：关掉 SQL 语义检查（扇出、关联条件不符合外键），只答原问法

三者的题数不同，下面比较消融效果时只取原问法（`variant == "question"`）。

In [13]:
base_s, base_r = runs["baseline"]
q = base_r[base_r["variant"] == "question"]
by_cat = (
    base_r.assign(
        问法=base_r["variant"].map(lambda v: "原问法" if v == "question" else "换问法")
    )
    .pivot_table(index="category", columns="问法", values="passed", aggfunc="mean")
    .mul(100)
    .round(0)
)
for name in ("no_examples", "no_lint"):
    _, r = runs[name]
    by_cat[name] = r.groupby("category")["passed"].mean().mul(100).round(0)
by_cat.loc["整体"] = [
    q["passed"].mean() * 100,
    base_r[base_r["variant"] != "question"]["passed"].mean() * 100,
    runs["no_examples"][1]["passed"].mean() * 100,
    runs["no_lint"][1]["passed"].mean() * 100,
]
by_cat.round(0)

问法,原问法,换问法,no_examples,no_lint
category,,,,
activity,80.0,80.0,80.0,80.0
date_role,100.0,100.0,100.0,100.0
distinct,100.0,100.0,100.0,100.0
fanout,80.0,80.0,80.0,80.0
funnel,100.0,80.0,100.0,100.0
order_vs_pay,100.0,100.0,100.0,100.0
refund,100.0,100.0,100.0,100.0
region_role,100.0,100.0,100.0,100.0
relative_time,100.0,40.0,100.0,100.0


**怎么读这张表**

- **整体 91%**（91/100）：原问法 96%（48/50），换问法 86%（43/50）。换个说法就多错 5 道，说明模型对措辞敏感，问法鲁棒性是值得单独看的指标
- **按类别**：日期角色、地区角色、去重计数、退款口径全对；错误集中在“相对时间”“活动归因”“扇出”几类，但扇出类错的其实不是扇出（见下面的失败环节）
- **成本**：每题约 6 次 LLM 调用、约 2 万 tokens。第一次跑（没有缓存，4 题并行）平均每题约 105 秒，主要卡在 CPU 上的向量计算；上表里的耗时是用缓存回放的重跑，不代表真实延迟

**这个 91% 是修过判分器之后的数字。** 第一次跑出来是 89%，逐题看失败明细时发现 `order_vs_pay_02` 的两次作答其实是对的：模型按 `'2025-01'` 输出月份，标准答案按 `dim_date.month` 输出整数 1。多轮评测里同样的误判还有 4 次。于是在 `compare.py` 加了第 6 条规则（第 4 节），重跑 `check_dataset` 和 `--predictor trap` 确认没有把任何错误写法判对，再用缓存回放重新判分。**评测本身也要被评测**：先看失败明细，再相信准确率。

### 失败出在哪一环

In [14]:
base_r[~base_r["passed"]].groupby(["stage"]).size().rename("次数").to_frame()

,次数
stage,
流程异常,1
结果不符,6
过滤误删,2


In [15]:
fails = base_r[~base_r["passed"]][
    [
        "case_id",
        "variant",
        "stage",
        "recall_missing",
        "filter_missing",
        "correct_attempts",
        "error",
    ]
]
fails.sort_values(["stage", "case_id"])

,case_id,variant,stage,recall_missing,filter_missing,correct_attempts,error
89,relative_time_05,paraphrase_1,流程异常,"[tables:fact_order, tables:dim_date, metrics:支付金额]","[tables:fact_order, tables:dim_date, metrics:支付金额]",0,OutputParserException: Invalid json output: [\n “销售额”，\n “销售额同比增长率”，\n “同比增长率”，\n “同比增速”，\n “同比增幅”，\n “销售收入”，\...
16,fanout_04,question,结果不符,[],[],0,NaN
17,fanout_04,paraphrase_1,结果不符,[],[],0,NaN
95,funnel_03,paraphrase_1,结果不符,[],[],0,NaN
83,relative_time_02,paraphrase_1,结果不符,[],[],0,NaN
87,relative_time_04,paraphrase_1,结果不符,[],[],0,NaN
55,snowflake_03,paraphrase_1,结果不符,[],[],0,NaN
70,activity_01,question,过滤误删,[],[tables:fact_order],0,NaN
71,activity_01,paraphrase_1,过滤误删,[],[tables:fact_order],0,NaN


**逐条看失败明细，比看准确率更有用。** 9 次失败可以归成几类，大部分都能转成具体的改进：

| 失败 | 真实原因 | 改进方向 |
| --- | --- | --- |
| `fanout_04`（两种问法）、`snowflake_03` 换问法 | “销量”按下单日期统计、没有限定已支付。业务术语只定义了“销售额”，没有“销量” | 在 `meta_config.yaml` 的 `terms` 加“销量 / 卖了多少件”= 已支付订单的件数 |
| `relative_time_02`、`_04` 换问法 | SQL 里用了 `CURDATE()`（真实日期 2026 年），没用注入的“今天” 2025-12-31 | 语义检查加一条规则：禁止 `CURDATE()` / `NOW()`，交给 `correct_sql` 修正 |
| `activity_01`（两种问法） | 过滤节点删掉了 `fact_order`，模型无法加“已支付”条件 | 过滤 Prompt 强调：统计金额 / 件数的问题要保留订单表以便判断是否支付 |
| `relative_time_05` 换问法 | 模型在 JSON 里输出全角引号 “” 和全角逗号，解析失败，整条流程中断 | 检索词扩展失败时降级为不扩展，而不是中断（已知问题：JSON 解析没有容错） |
| `funnel_03` 换问法 | 模型按指标公式“支付买家数 / 浏览用户数”计算；标准答案只统计“浏览过的人里有多少付款”。`meta_config.yaml` 里这个指标的描述前后矛盾 | 统一指标定义和标准答案的口径：**这是评测帮我们找到的知识库 bug，不是模型的错** |

**归因的局限也在这里体现出来**：`fanout_04` 被归为“结果不符”，看 SQL 才知道错的是支付口径而不是扇出；之前 `order_vs_pay_02` 被归为“召回遗漏 dim_date”，其实这题根本不需要 dim_date（`expect` 把一种写法需要的表当成了必需）。归因只是第一步，还是要回到 SQL。

### 消融：去掉一个模块，哪些题由对变错

In [16]:
def flips(name):
    """与基线原问法相比，消融后由对变错、由错变对的题"""
    _, r = runs[name]
    merged = (
        q.set_index("case_id")["passed"]
        .to_frame("基线")
        .join(r.set_index("case_id")["passed"].rename(name))
    )
    return {
        "由对变错": sorted(merged.index[merged["基线"] & ~merged[name]]),
        "由错变对": sorted(merged.index[~merged["基线"] & merged[name]]),
    }


pd.DataFrame({name: flips(name) for name in ("no_examples", "no_lint")}).T

,由对变错,由错变对
no_examples,[],[]
no_lint,[],[]


两组消融的结果都和基线一样（原问法 96%），**但这不说明这两个模块没用**，要结合缓存命中数看：

- **关掉已验证示例**：第一次跑时只有 17 次调用的提示词发生变化，说明只有少数题召回到了示例（`knowledge.example_score_threshold=0.75` 偏高，或示例与评测题的问法差得远），而召回到示例的题，在没有示例时也答对了。在这套题上，示例既没帮忙也没添乱；它们的价值要在更难、与示例写法更接近的题上才能体现
- **关掉语义检查**：基线里有 6 次作答被语义检查拦下修正过（`refund_03`、`refund_05`、`date_role_04`、`funnel_03`），关掉之后这几题未经修正的 SQL 也是对的。原因是检查发现的是“退款表关联订单明细后对明细金额求和”的扇出风险，而**这份数据里每条订单明细最多只有一条退款**，扇出在数据上从来不会真正放大金额。风险是真实的，只是评测数据没有覆盖到

这是消融实验最常见的陷阱：**评测集没有覆盖某种情况，消融就测不出对应模块的价值**。改进办法是补题和补数据：让生成器产生“一条明细多次部分退款”的数据，并为它出一道陷阱题，再重跑这组消融。

**关于可复现**：重跑时仍有 2.6% 的调用没命中缓存，8 次作答的 SQL 和第一次不同（对错没有变化）。原因是 TEI 会把并发请求合在一个批次里计算，批次不同，同一段文本的向量会有约 1e-7 的差异，近似相同的召回结果因此可能换了顺序。要做到完全可复现，还需要把向量也缓存起来。

### 看几道失败题的 SQL

In [17]:
gold_by_id = {i["id"]: i["turns"][0]["gold_sql"].strip() for i in traps}
trap_by_id = {i["id"]: i["turns"][0]["trap"] for i in traps}
for _, row in q[~q["passed"]].head(4).iterrows():
    print(f"=== {row['case_id']}（{row['stage']}）{row['question']}")
    print(f"陷阱：{trap_by_id[row['case_id']]}")
    print("--- 模型 SQL\n" + (row["sql"] or f"（无 SQL）{row['error']}"))
    print("--- 标准 SQL\n" + gold_by_id[row["case_id"]] + "\n")

=== fanout_04（结果不符）2025 年各品牌卖出了多少件商品？
陷阱：用订单级的总件数按明细关联后求和，件数被重复计算且被算到所有品牌上
--- 模型 SQL
SELECT s.brand_name AS `品牌`, SUM(oi.quantity) AS `销量`
FROM fact_order_item oi
JOIN fact_order o ON oi.order_id = o.order_id
JOIN dim_sku s ON oi.sku_id = s.sku_id
WHERE o.order_date_id BETWEEN 20250101 AND 20251231
GROUP BY s.brand_name
ORDER BY `销量` DESC
--- 标准 SQL
select k.brand_name as 品牌, sum(i.quantity) as 销量
from fact_order_item i
join fact_order o on i.order_id = o.order_id
join dim_sku k on i.sku_id = k.sku_id
where o.pay_time is not null
  and o.pay_date_id between 20250101 and 20251231
group by k.brand_name

=== activity_01（过滤误删）2025 年双 11 活动的商品一共卖了多少件？
陷阱：用活动期间代替活动归因，把期间内未参加活动的商品也算了进去
--- 模型 SQL
SELECT SUM(foi.quantity) AS 销量
FROM fact_order_item foi
INNER JOIN dim_activity da ON foi.activity_id = da.activity_id
WHERE da.activity_name = '2025年双11'
--- 标准 SQL
select sum(i.quantity) as 销量
from fact_order_item i
join fact_order o on i.order_id = o.order_id
join dim_activity a on i.activity_id 

## 10. 多轮评测

多轮题的每一轮都有标准 SQL，第 2 轮起还标注了期望意图（`gold_intent`）和期望的改写结果（`gold_standalone`）。两种模式：

| 模式 | 历史从哪来 | 衡量什么 |
| --- | --- | --- |
| `teacher`（teacher forcing） | 每轮新开会话，历史用前几轮的**标准答案**填充（标准改写、标准 SQL 及其结果预览） | 单轮追问能力：前一轮答错不会连累后面，每一轮都是公平的考题 |
| `free`（自由对话） | 同一个会话依次作答，历史是**模型自己的** | 真实体验：前一轮的错误会传递到后面 |

指标：意图准确率（第 2 轮起）、追问轮准确率、全部轮次准确率、整段对话成功率（每一轮都对）。多轮的归因在单轮之前多一个“意图错误”。

多轮功能是分两步上线的，这里正好用评测比较前后两版：

- **第一阶段**（`*_phase1`）：只有追问改写，结果追问也改写后重新查库，不会反问；评测集当时是 20 段对话
- **第 4、5 步之后**（`mt_teacher` / `mt_free`）：追问带上一轮 SQL 作参考、结果追问直接基于上一轮结果回答、信息不足时反问；评测集增加了 2 段需要反问的对话（共 22 段），结果追问轮增加了回答关键内容 `gold_answer_contains`

两版的题目数不同，比较时看同一批对话（前 20 段）的追问轮准确率最公平。失败明细里会并列预测改写和标准改写，方便判断是改写错还是 SQL 错；直接回答的轮次在说明里显示回答内容。

In [18]:
mt = {
    name: load_run(name)
    for name in ("mt_teacher_phase1", "mt_free_phase1", "mt_teacher", "mt_free")
}
first20 = {i["id"] for i in multiturn[:20]}  # 两版都有的 20 段对话


def followup_rate(r):
    """同一批 20 段对话里，第 2 轮及以后的准确率"""
    f = r[(r["turn"] > 1) & r["case_id"].isin(first20)]
    return f"{f['passed'].mean():.0%}（{int(f['passed'].sum())}/{len(f)}）"


pd.DataFrame(
    {
        name: {
            "对话数": s["dialogs"],
            "整段对话成功率": f"{s['dialog_success_rate']:.0%}（{s['dialog_success']}/{s['dialogs']}）",
            "追问轮准确率（同一批 20 段）": followup_rate(r),
            "追问轮准确率（全部）": f"{s['followup_accuracy']:.0%}（{s['followup_passed']}/{s['followup_total']}）",
            "意图准确率": f"{s['intent_accuracy']:.0%}"
            if s["intent_accuracy"] is not None
            else "-",
            "意图混淆": s["intent_confusion"] or "无",
            "直接回答 / 反问轮数": f"{s.get('answered_turns', 0)} / {s.get('clarify_turns', 0)}",
            "每轮平均 LLM 调用": s["avg_llm_calls"],
        }
        for name, (s, r) in mt.items()
    }
)

,mt_teacher_phase1,mt_free_phase1,mt_teacher,mt_free
对话数,20,20,22,22
整段对话成功率,85%（17/20）,80%（16/20）,86%（19/22）,91%（20/22）
追问轮准确率（同一批 20 段）,92%（22/24）,88%（21/24）,100%（24/24）,100%（24/24）
追问轮准确率（全部）,92%（22/24）,88%（21/24）,92%（24/26）,96%（25/26）
意图准确率,100%,100%,92%,92%
意图混淆,无,无,"{'followup→clarify': 1, 'clarify→followup': 1}","{'result_qa→followup': 1, 'clarify→followup': 1}"
直接回答 / 反问轮数,0 / 0,0 / 0,2 / 2,1 / 1
每轮平均 LLM 调用,6.52,6.52,6.33,6.44


In [19]:
pd.DataFrame(
    {
        name: {
            cat: f"{v['followups_passed']}/{v['followups']}"
            for cat, v in s["by_category"].items()
        }
        for name, (s, _) in mt.items()
        if not name.endswith("phase1")
    }
).rename_axis("追问类别")

,mt_teacher,mt_free
追问类别,,
add_filter,4/4,4/4
change_dim,5/5,5/5
clarify,0/2,1/2
correction,3/3,3/3
ellipsis,6/6,6/6
result_ref,4/4,4/4
topic_switch,2/2,2/2


In [20]:
_, mt_r = mt["mt_free"]
mt_r[~mt_r["passed"]][
    [
        "case_id",
        "turn",
        "stage",
        "gold_intent",
        "intent",
        "standalone",
        "gold_standalone",
        "answer",
    ]
]

,case_id,turn,stage,gold_intent,intent,standalone,gold_standalone,answer
32,mt_result_ref_03,1,流程异常,new,new,2025 年各支付方式的支付订单数,2025 年各支付方式的支付订单数,NaN
45,mt_clarify_01,2,意图错误,clarify,followup,最近华南大区的支付订单数是多少？,最近 7 天华南大区的支付订单数是多少？,NaN


**第 4、5 步前后对比（同一批 20 段对话的追问轮）：teacher 92% → 100%，自由对话 88% → 100%。** 第一阶段的 3 种真实错误都被修掉了，而且能对上原因：

| 第一阶段的错误 | 现在 | 起作用的改动 |
| --- | --- | --- |
| `mt_add_filter_01` 第 2 轮：加了“钻石会员”条件后，地区改用了用户常住地 | 沿用上一轮 SQL 的收货地关联 | 追问带上一轮 SQL 作参考 |
| `mt_correction_02` 第 2 轮：“销量”按下单口径统计 | 沿用上一轮的已支付条件 | 追问带上一轮 SQL 作参考。对照单轮评测：同样的“销量”单独问时仍会答错，参考 SQL 把口径带过来了 |
| `mt_change_dim_01` 第 3 轮：“再按平台拆一下”丢了“每个月” | 按月和平台一起统计 | 改写规则区分“叠加维度”和“替换维度” |

**结果追问**：`mt_result_ref_01`、`_03` 的“排第一的是哪个”“哪种最少”直接基于上一轮结果回答，没有查库，回答里的占比也算对了（华东 14577500.76 / 46432548.78 = 31.40%）。自由对话里 `mt_result_ref_03` 第 1 轮查询失败，第 2 轮没有可用的上一轮结果，模型改判为追问并重新查库，同样答对：这是设计好的退路。

**反问是现在最弱的一环（2 段对话里只成功 1 次）**：

- `mt_clarify_02`（“那个品牌”有三个候选）：两种模式都正确反问了，扮演用户回答“苹果”后，自由对话答对；teacher 模式在后面的 JSON 解析上出错
- `mt_clarify_01`（“那最近的呢”）：两种模式都没有反问，直接把“最近”带进了问题，属于该问没问
- 反过来，teacher 模式下 `mt_ellipsis_03` 第 3 轮“它的件单价”不该问却问了（上文有小米和耐克两个品牌，规则是默认承接最近一轮），扮演用户回答“按你的理解直接查询”后答对，但多打扰了用户一次

该问没问会答错，不该问却问了会打扰用户，两种错误都要统计。意图准确率（92%）和意图混淆表把它们分开显示出来。

**剩下的失败里，JSON 解析占了一半**：单轮 1 次、多轮 teacher 2 次、自由对话 1 次，都是模型在 JSON 里输出了全角引号或全角逗号，整条流程中断。这是下一步收益最大的改进：给 `JsonOutputParser` 加一层容错（把全角符号换成半角再解析，检索词扩展失败时降级为不扩展），不需要改提示词，用缓存回放就能验证效果。

## 11. 动手实验建议

按“改一处、看指标怎么变”的思路，每个实验都有明确要回答的问题：

1. **问法鲁棒性**：比较原问法和换问法的准确率（第 9 节已有）。差距大说明模型依赖特定措辞，可以加别名或业务术语
2. **few-shot 的真实贡献**：`--no-examples` 已经做了；进一步可以只删掉某一条示例，看哪些题受影响，判断示例是在教“写法”还是在“泄题”
3. **召回阈值**：改 `conf/app_config.yaml` 的 `qdrant.score_threshold`（0.6 → 0.5 / 0.7），看召回遗漏和 token 数怎么变
4. **换模型**：改 `llm.model_name`，用同一份评测集比较准确率和成本；用 `--baseline` 看哪些题由对变错
5. **加一道题**：在 `traps.yaml` 加题，先跑 `check_dataset` 证明错误写法可区分，再跑 `--predictor gold` / `trap` 自检
6. **判分规则的边界**：在第 4 节的单元里构造更多用例，想想“允许多出列”会不会放过错误答案

每次实验都加 `--llm-cache`，没变化的调用直接回放；用 `--baseline` 对照基线，退化时一眼就能看出来。

## 附录：Windows + Docker 与 Linux 的差异

本仓库的命令和这本 notebook 都以 Linux（bash）为准，在 Linux 云环境验证过。在 Windows + Docker Desktop 上复现时注意：

| 项目 | Linux（本仓库） | Windows + Docker Desktop |
| --- | --- | --- |
| 命令行里临时设环境变量 | `LLM_API_KEY=xxx uv run ...` | PowerShell：`$env:LLM_API_KEY="xxx"`；最省事是写进项目根目录的 `.env`，代码启动时会自动读取 |
| 导入 SQL 快照 | `gunzip -c docker/snapshot/dw_snapshot.sql.gz \| docker exec -i mysql mysql -uroot -pdili123` | PowerShell 的管道和 `<` 重定向不适合二进制文件，可在 Git Bash / WSL 里执行同一条命令，或先 `docker cp docker/snapshot/dw_snapshot.sql.gz mysql:/tmp/`，再 `docker exec mysql sh -c "gunzip -c /tmp/dw_snapshot.sql.gz \| mysql -uroot -pdili123"` |
| 输出重定向到文件 | 无需处理 | Windows 默认 GBK 编码，runner 输出里的 ✓ ✗ 写入文件时可能报 `UnicodeEncodeError`，先设 `$env:PYTHONUTF8="1"` |
| TEI 镜像 | `linux/amd64` 原生运行 | x64 机器上一样；ARM 机器会走模拟，非常慢 |
| TEI 并发上限 | compose 里 `MAX_CONCURRENT_REQUESTS=512` | 同一个镜像，同样需要（原来的 16 会让知识库构建时 TEI panic 重启） |
| 内存 | 全套服务实测约 6～8 GB（TEI 的 bge-large 约 2～3 GB，ES 约 1～2 GB） | Docker Desktop（WSL2）默认内存可能不够，在 `.wslconfig` 里调到 8 GB 以上 |
| 3306 端口 | 一般空闲 | 本机装过 MySQL 服务时会冲突，先停掉本机服务或改 compose 端口映射 |
| 数据卷初始化 | MySQL 账号由 compose 的 `MYSQL_USER` 在首次初始化时创建 | 同上；改过账号要 `docker compose -f docker/docker-compose.yaml down -v` 重建 |
| 换行符 | 仓库统一 LF | Git 的 `core.autocrlf=true` 会把文件检出成 CRLF。项目读取 SQL、Prompt、YAML 时都按行处理，不受影响 |
| 文件编码 | — | 项目代码读写文件都显式用了 UTF-8，不依赖系统默认编码 |
| 异步事件循环 | 默认的 selector 循环 | Windows 默认是 Proactor 循环，asyncmy、aiohttp 一般可用，**未在 Windows 上实测** |
| 结果可复现性 | LLM `temperature=0`，加上缓存回放后完全可复现 | 不同机器上 TEI 的浮点计算可能有微小差异，召回结果的顺序可能不同，导致提示词变化、缓存不命中、结果略有出入 |